# Ch.3 — XGBoost, LightGBM, CatBoost

## EnsembleAI Challenge — Ch.3

Industrial-strength gradient boosting frameworks.
Compare **XGBoost** (2nd-order optimization), **LightGBM** (histogram + GOSS), **CatBoost** (ordered boosting).

Goal: best RMSE with reasonable training time.

## Setup & Data

In [ ]:
# ── Setup & Data ─────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import time
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

IMG = "img/"
import os; os.makedirs(IMG, exist_ok=True)

np.random.seed(42)

data = fetch_california_housing()
X, y = data.data, data.target
feature_names = list(data.feature_names)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
X_tr2, X_val, y_tr2, y_val = train_test_split(X_tr, y_tr, test_size=0.15, random_state=42)

print(f"Train: {len(X_tr2):,}  Val: {len(X_val):,}  Test: {len(X_te):,}")

## XGBoost

In [ ]:
# ── XGBoost ──────────────────────────────────────────────────────────────────
from xgboost import XGBRegressor

t0 = time.time()
xgb = XGBRegressor(
    n_estimators=1000, learning_rate=0.05, max_depth=4,
    subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
    early_stopping_rounds=30, eval_metric='rmse',
    random_state=42, verbosity=0)
xgb.fit(X_tr2, y_tr2, eval_set=[(X_val, y_val)], verbose=False)
t_xgb = time.time() - t0

rmse_xgb = np.sqrt(mean_squared_error(y_te, xgb.predict(X_te)))
print(f"XGBoost — RMSE: {rmse_xgb:.4f}  Time: {t_xgb:.2f}s  Best round: {xgb.best_iteration}")

## LightGBM

In [ ]:
# ── LightGBM ─────────────────────────────────────────────────────────────────
import lightgbm as lgb

t0 = time.time()
lgbm = lgb.LGBMRegressor(
    n_estimators=1000, learning_rate=0.05, num_leaves=31,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, verbosity=-1)
lgbm.fit(X_tr2, y_tr2, eval_set=[(X_val, y_val)],
         callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
t_lgb = time.time() - t0

rmse_lgb = np.sqrt(mean_squared_error(y_te, lgbm.predict(X_te)))
print(f"LightGBM — RMSE: {rmse_lgb:.4f}  Time: {t_lgb:.2f}s  Best round: {lgbm.best_iteration_}")

## CatBoost

In [ ]:
# ── CatBoost ──────────────────────────────────────────────────────────────────
from catboost import CatBoostRegressor

t0 = time.time()
cat = CatBoostRegressor(
    iterations=1000, learning_rate=0.05, depth=6,
    l2_leaf_reg=3, random_seed=42, verbose=0)
cat.fit(X_tr2, y_tr2, eval_set=(X_val, y_val), early_stopping_rounds=30)
t_cat = time.time() - t0

rmse_cat = np.sqrt(mean_squared_error(y_te, cat.predict(X_te)))
print(f"CatBoost — RMSE: {rmse_cat:.4f}  Time: {t_cat:.2f}s  Best round: {cat.best_iteration_}")

## Framework Comparison

In [ ]:
# ── Comparison Chart ─────────────────────────────────────────────────────────
names  = ['XGBoost', 'LightGBM', 'CatBoost']
rmses  = [rmse_xgb, rmse_lgb, rmse_cat]
times  = [t_xgb, t_lgb, t_cat]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = ['#2171b5', '#41ab5d', '#e6550d']

axes[0].bar(names, rmses, color=colors, edgecolor='white')
axes[0].set_ylabel('Test RMSE'); axes[0].set_title('Accuracy (lower=better)', fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(names, times, color=colors, edgecolor='white')
axes[1].set_ylabel('Training time (s)'); axes[1].set_title('Speed (lower=better)', fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('XGBoost vs LightGBM vs CatBoost — California Housing', fontsize=12, fontweight='bold')
plt.tight_layout()
fig.savefig(f'{IMG}ch03_framework_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n{'Framework':<12} {'RMSE':>8} {'Time(s)':>8} {'Rounds':>8}")
print("-" * 40)
for n, r, t, rounds in zip(names, rmses, times,
                            [xgb.best_iteration, lgbm.best_iteration_, cat.best_iteration_]):
    print(f"{n:<12} {r:>8.4f} {t:>8.2f} {rounds:>8}")

## XGBoost Learning Curve

In [ ]:
# ── XGBoost Validation Curve ─────────────────────────────────────────────────
val_rmse = xgb.evals_result()['validation_0']['rmse']

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(val_rmse, color='steelblue', linewidth=1.5)
ax.axvline(xgb.best_iteration, color='coral', linestyle='--',
           label=f'Early stop at round {xgb.best_iteration}')
ax.set_xlabel('Boosting round'); ax.set_ylabel('Validation RMSE')
ax.set_title('XGBoost — Validation RMSE Across Rounds', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(f'{IMG}ch03_xgb_learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## Feature Importances: XGBoost vs LightGBM

In [ ]:
# ── Feature Importances Comparison ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, model, title in [
    (axes[0], xgb, 'XGBoost'),
    (axes[1], lgbm, 'LightGBM'),
]:
    imps = model.feature_importances_
    order = np.argsort(imps)
    ax.barh([feature_names[i] for i in order], imps[order],
            color='steelblue', edgecolor='white')
    ax.set_xlabel('Importance'); ax.set_title(title, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)

plt.suptitle('Feature Importances — XGBoost vs LightGBM', fontsize=12, fontweight='bold')
plt.tight_layout()
fig.savefig(f'{IMG}ch03_feature_importances.png', dpi=150, bbox_inches='tight')
plt.show()

## Progress Check

| # | Constraint | Status | Evidence |
|---|-----------|--------|----------|
| 1 | IMPROVEMENT | ✅ | All three beat sklearn GB |
| 2 | DIVERSITY | ✅ | Different optimization strategies |
| 3 | EFFICIENCY | ✅ | LightGBM: fastest training; all: fast inference |
| 4 | INTERPRETABILITY | ⚡ | Feature importance; SHAP in Ch.4 |
| 5 | ROBUSTNESS | ✅ | Regularization + early stopping |

**Next**: Ch.4 — SHAP explains per-prediction feature contributions.

## Exercises

1. **XGBoost regularization**: Train XGBoost with `reg_lambda` in `[0, 0.1, 1, 10, 100]`. Plot test RMSE. How much does regularization help?
2. **LightGBM num_leaves**: Train with `num_leaves` in `[15, 31, 63, 127, 255]`. When does leaf-wise growth overfit?
3. **Speed benchmark on larger data**: Repeat the 3-framework comparison using `make_regression(n_samples=100000)`. Which framework benefits most from larger data?

In [ ]:
# ── Exercise 1: XGBoost regularization sweep ────────────────────────────────
# TODO: for lam in [0, 0.1, 1, 10, 100]:
#     xgb_r = XGBRegressor(reg_lambda=lam, ...)
#     fit with early stopping, record RMSE
pass

In [ ]:
# ── Exercise 2: LightGBM num_leaves sweep ───────────────────────────────────
# TODO: for nl in [15, 31, 63, 127, 255]:
#     lgb_nl = lgb.LGBMRegressor(num_leaves=nl, ...)
#     fit with early stopping, record RMSE
pass

In [ ]:
# ── Exercise 3: Large data speed benchmark ──────────────────────────────────
# TODO: from sklearn.datasets import make_regression
#     X_big, y_big = make_regression(n_samples=100_000, n_features=20, random_state=42)
#     Time each framework with early stopping
pass